# Детекция бабочек: OWLv2 base и YOLOv8x

В Kaggle включить **T4 ×2** и **Internet**. Загрузить `butterfly_dataset.zip` как один Dataset, добавить его в Input и выполнить Run All. Kaggle распакует ZIP при загрузке.

В наборе **40 фото с эталонной разметкой** и **400 фото для разметки**. Обе модели обрабатывают все 440 фото. Метрики считаем только на 40 фото с эталоном. Для остальных 400 сохраняем предсказания отдельно.

В ячейке настроек можно изменить `BATCH_SIZE` — число фотографий в батче на одной GPU. По умолчанию 1. Для каждой модели показываем прогресс по всем 440 фото, без прогрева.

Для обеих моделей используем вход **960×960**, исходное фото и горизонтальное отражение, NMS каждого вида, подтверждение рамки двумя видами и среднее координат с весами score, затем финальный NMS. IoU сопоставления видов — **0,35**. Подготовка пикселей — штатная для каждой модели.

| Модель | Confidence | NMS каждого вида | Финальный NMS |
|---|---:|---:|---:|
| OWLv2 base | 0.25 | 0.5 | 0.5 |
| YOLOv8x-OIV7 | 0.45 | 0.7 | 0.8 |

Запросы OWL одинаковы для всех фото. Из первого запроса получаем рамки бабочек; остальные помогают отсеивать похожие объекты. Минимальная разница логитов — **−0,5**. Эталон и категории источников моделям не передаём.

Фотографии берём только из загруженного набора. Веса моделей скачиваются через интернет.

Каждый запуск заново обрабатывает все 440 фото. Эталон читаем после предсказаний. Precision, recall и число ошибок FP+FN считаем при IoU ≥ 0,5. Ложные срабатывания на негативах — число рамок на 16 фото без бабочек из размеченного набора.

Время в таблице — сумма времени батчей с 40 размеченными фото: подготовка входа, модель и обработка рамок. Размеченные и неразмеченные фото обрабатываем отдельными батчами. Чтение файлов, загрузку весов и прогрев не учитываем. На каждую T4 приходится 220 фото. Предсказания в JSONL для двух наборов, настройки и таблица метрик сохраняются в ZIP.


In [1]:
# Установка
import subprocess
import sys

packages = [
    'transformers==4.57.6',
    'tokenizers==0.22.2',
    'huggingface-hub==0.36.2',
    'ultralytics==8.4.208',
    'pandas>=2.2,<3',
    'Pillow==11.2.1',
    'tqdm>=4.66,<5',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', *packages])
subprocess.check_call([sys.executable, '-c', 'from ultralytics import YOLO'])


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 74.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 52.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 13.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 34.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 57.9 MB/s  0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 13.0.0
    Uninstalling pillow-13.0.0:
      Successfully uninstalled pillow-13.0.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    Uninstalling tokenizers-0.23.3:
      Successfully uninstalled tokenizers-0.23.3
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


0

In [3]:
# Настройки запуска
MIN_CONFIDENCE = 0.25

OWL = {'confidence': 0.25,
 'nms_iou': 0.5,
 'fusion': {'iou': 0.35, 'min_views': 2, 'nms_iou': 0.5},
 'queries': ['a single butterfly',
             'a photo of a moth',
             'a photo of a caterpillar',
             'a photo of a chrysalis',
             'a photo of a dragonfly',
             'a photo of a fly',
             'a photo of a bee',
             'a photo of a beetle',
             'a photo of a flower',
             'a flower petal'],
 'negative_margin': -0.5}

YOLO_SETTINGS = {'confidence': 0.45, 'nms_iou': 0.7, 'fusion': {'iou': 0.35, 'min_views': 2, 'nms_iou': 0.8}}



In [4]:
# Импорты и пути
import gc
import json
import os
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from urllib.request import urlretrieve

os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['YOLO_AUTOINSTALL'] = 'false'

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from huggingface_hub import snapshot_download
from IPython.display import FileLink, display
from transformers import AutoProcessor, Owlv2ForObjectDetection
from tqdm.auto import tqdm
from ultralytics import YOLO

# Используем FP32 для обеих моделей.
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/butterflies_final')
OUT.mkdir(exist_ok=True)




In [5]:
# Данные
def save(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False))


def load_data():
    manifests = list(INPUT.rglob('photos.json'))
    assert len(manifests) == 1, 'Добавьте один Dataset из butterfly_dataset.zip в Kaggle Input.'
    directory = manifests[0].parent
    manifest = json.loads(manifests[0].read_text())
    photos = []
    for subset, expected in [('labeled', 40), ('unlabeled', 400)]:
        assert len(manifest[subset]) == expected, subset
        for row in manifest[subset]:
            relative = Path(row['file'])
            assert relative.parts[:2] == ('images', subset) and '..' not in relative.parts, row['file']
            image_path = directory / relative
            assert image_path.is_file(), image_path.name
            photos.append({'photo_id': row['photo_id'], 'subset': subset, 'path': image_path})
    assert len(photos) == len({row['photo_id'] for row in photos}) == 440
    return photos, directory




In [6]:
# Веса
def load_weights(directory):
    metadata = json.loads((directory / 'models.json').read_text())
    owl_files = metadata['owl']['files']
    weights = {'owl': {key: metadata['owl'][key] for key in ['repo', 'revision']},
               'yolo': {key: metadata['yolo'][key] for key in ['file', 'url']}}
    owl = OUT / 'owlv2'
    snapshot_download(weights['owl']['repo'], revision=weights['owl']['revision'],
                      allow_patterns=[Path(name).name for name in owl_files], local_dir=owl)
    assert all((owl / Path(name).name).is_file() for name in owl_files)
    yolo = OUT / weights['yolo']['file']
    if not yolo.is_file():
        urlretrieve(weights['yolo']['url'], yolo)
    assert yolo.is_file()
    return owl, yolo, weights




In [7]:
# Общие функции для рамок
def pairwise_iou(a, b):
    low = np.maximum(a[:, None, :2], b[None, :, :2])
    high = np.minimum(a[:, None, 2:], b[None, :, 2:])
    intersection = np.prod(np.maximum(high - low, 0), axis=2)
    area_a = np.prod(a[:, 2:] - a[:, :2], axis=1)
    area_b = np.prod(b[:, 2:] - b[:, :2], axis=1)
    return intersection / np.maximum(area_a[:, None] + area_b[None, :] - intersection, 1e-12)


def nms(boxes, scores, threshold):
    order = np.argsort(-scores, kind='stable')
    keep = []
    while len(order):
        i = order[0]
        keep.append(i)
        order = order[1:]
        overlap = pairwise_iou(boxes[i:i + 1], boxes[order])[0]
        order = order[overlap <= threshold]
    return np.asarray(keep, dtype=int)


def fuse_views(boxes_by_view, scores_by_view, threshold, min_views):
    candidates = []
    for view, (boxes, scores) in enumerate(zip(boxes_by_view, scores_by_view)):
        for box, score in zip(boxes, scores):
            candidates.append((float(score), np.asarray(box, dtype=float), view))
    candidates.sort(key=lambda row: -row[0])
    clusters = []
    for score, box, view in candidates:
        possible = [i for i, c in enumerate(clusters) if view not in c['views']]
        overlaps = pairwise_iou(box[None], np.asarray([clusters[i]['box'] for i in possible]).reshape(-1, 4))[0]
        best = int(np.argmax(overlaps)) if len(overlaps) else None
        if best is not None and overlaps[best] >= threshold:
            cluster = clusters[possible[best]]
            weight = cluster['weight'] + score
            cluster['box'] = (cluster['box'] * cluster['weight'] + box * score) / weight
            cluster['weight'] = weight
            cluster['views'].add(view)
        else:
            clusters.append({'box': box.copy(), 'weight': score, 'views': {view}})
    clusters = [c for c in clusters if len(c['views']) >= min_views]
    clusters.sort(key=lambda c: -(c['weight'] / len(c['views'])))
    boxes = np.asarray([c['box'] for c in clusters]).reshape(-1, 4)
    scores = np.asarray([c['weight'] / len(c['views']) for c in clusters])
    return (boxes, scores)


def to_ltwh(boxes):
    result = []
    for box in boxes:
        x1, y1, x2, y2 = map(float, box)
        assert 0 <= x1 < x2 <= 1 and 0 <= y1 < y2 <= 1
        result.append({'left': x1, 'top': y1, 'width': x2 - x1, 'height': y2 - y1})
    return result


def merge_views(boxes_by_view, scores_by_view, config):
    fusion = config['fusion']
    boxes, scores = fuse_views(boxes_by_view, scores_by_view, fusion['iou'], fusion['min_views'])
    keep = nms(boxes, scores, fusion['nms_iou'])
    return to_ltwh(boxes[keep].clip(0, 1))


def open_image(row):
    with Image.open(row['path']) as image:
        return ImageOps.exif_transpose(image).convert('RGB')




In [8]:
# OWLv2
def owl_boxes(output, image, flipped, confidence, index):
    logits = output.logits[index].float().cpu().numpy()
    boxes = output.pred_boxes[index].float().cpu().numpy()
    valid = 1 / (1 + np.exp(-logits[:, 0])) > confidence
    logits, boxes = logits[valid], boxes[valid]
    boxes = np.concatenate([boxes[:, :2] - boxes[:, 2:] / 2,
                            boxes[:, :2] + boxes[:, 2:] / 2], axis=1)
    width, height = image.size
    # Процессор дополняет кадр до квадрата, поэтому масштаб общий для двух осей.
    boxes *= max(width, height)
    boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, width)
    boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, height)
    if flipped:
        boxes[:, [0, 2]] = width - boxes[:, [2, 0]]
    boxes /= np.array([width, height, width, height], dtype=np.float32)
    valid = (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])
    return boxes[valid], logits[valid]


@torch.inference_mode()
def predict_owl(model, processor, images, device, config):
    boxes = [[] for _ in images]
    scores = [[] for _ in images]
    for flipped in [False, True]:
        views = [ImageOps.mirror(image) for image in images] if flipped else images
        inputs = processor(images=views, text=[config['queries']] * len(images), return_tensors='pt').to(device)
        assert tuple(inputs['pixel_values'].shape) == (len(images), 3, 960, 960)
        output = model(**inputs)
        for i, image in enumerate(images):
            b, logits = owl_boxes(output, image, flipped, config['confidence'], i)
            valid = logits[:, 0] - logits[:, 1:].max(axis=1) >= config['negative_margin']
            b, s = b[valid], 1 / (1 + np.exp(-logits[valid, 0]))
            keep = nms(b, s, config['nms_iou'])
            boxes[i].append(b[keep])
            scores[i].append(s[keep])
    return [merge_views(b, s, config) for b, s in zip(boxes, scores)]




In [9]:
# YOLOv8x
def yolo_boxes(result, image, flipped):
    boxes = result.boxes.xyxy.float().cpu().numpy().copy()
    scores = result.boxes.conf.float().cpu().numpy()
    boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, image.width)
    boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, image.height)
    boxes /= np.array([image.width, image.height, image.width, image.height], dtype=np.float32)
    if flipped:
        boxes[:, [0, 2]] = 1 - boxes[:, [2, 0]]
    valid = (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])
    return boxes[valid], scores[valid]


@torch.inference_mode()
def predict_yolo(model, images, device, config):
    boxes = [[] for _ in images]
    scores = [[] for _ in images]
    for flipped in [False, True]:
        views = [ImageOps.mirror(image) for image in images] if flipped else images
        results = model.predict(views, device=device, imgsz=960, rect=False, classes=[75],
            conf=config['confidence'], iou=config['nms_iou'], max_det=1000, quantize=32,
            nms=True, batch=len(images), augment=False, save=False, verbose=False)
        assert len(results) == len(images)
        for i, (result, image) in enumerate(zip(results, images)):
            b, s = yolo_boxes(result, image, flipped)
            boxes[i].append(b)
            scores[i].append(s)
    return [merge_views(b, s, config) for b, s in zip(boxes, scores)]




In [10]:
# Запуск моделей
def run(name, config, photos, owl_path, yolo_path, batch_size):
    models, processors = [], []
    # Загружаем по очереди: HF временно меняет общий dtype.
    for gpu in range(2):
        device = torch.device(f'cuda:{gpu}')
        if name.startswith('yolo'):
            model = YOLO(yolo_path).to(device)
            processor = None
            assert model.names[75].lower() == 'butterfly'
        else:
            processor = AutoProcessor.from_pretrained(owl_path, use_fast=False, local_files_only=True)
            model = Owlv2ForObjectDetection.from_pretrained(owl_path, local_files_only=True, dtype=torch.float32).to(device).eval()
            tokens = processor.tokenizer(config['queries'], truncation=False)['input_ids']
            assert max(map(len, tokens)) <= model.config.text_config.max_position_embeddings, 'Слишком длинный запрос.'
        assert next(model.parameters()).dtype == torch.float32
        models.append(model)
        processors.append(processor)
    del model

    def worker(gpu):
        torch.cuda.set_device(gpu)
        device = torch.device(f'cuda:{gpu}')
        def predict(images):
            if name.startswith('yolo'):
                return predict_yolo(models[gpu], images, device, config)
            return predict_owl(models[gpu], processors[gpu], images, device, config)
        local_photos = photos[gpu::2]
        warmup = [row for row in local_photos if row['subset'] == 'labeled'][:batch_size]
        predict([open_image(row) for row in warmup])
        torch.cuda.synchronize(gpu)
        result = []
        for subset in ['labeled', 'unlabeled']:
            rows = [row for row in local_photos if row['subset'] == subset]
            for start in range(0, len(rows), batch_size):
                batch = rows[start:start + batch_size]
                try:
                    images = [open_image(row) for row in batch]
                    torch.cuda.synchronize(gpu)
                    started = time.perf_counter()
                    boxes = predict(images)
                    torch.cuda.synchronize(gpu)
                    elapsed = time.perf_counter() - started
                    assert len(boxes) == len(batch)
                    records = [{'photo_id': row['photo_id'], 'boxes': b, 'seconds': elapsed / len(batch), 'error': None}
                               for row, b in zip(batch, boxes)]
                except Exception as error:
                    records = [{'photo_id': row['photo_id'], 'boxes': None, 'seconds': 0,
                                'error': f'{type(error).__name__}: {error}'} for row in batch]
                result.extend(records)
                with progress.get_lock():
                    progress.update(len(batch))
        return result

    try:
        with tqdm(total=len(photos), desc=name, unit='фото') as progress:
            with ThreadPoolExecutor(2) as executor:
                result = [row for part in executor.map(worker, range(2)) for row in part]
        order = {row['photo_id']: i for i, row in enumerate(photos)}
        result.sort(key=lambda row: order[row['photo_id']])
        assert len(result) == len({row['photo_id'] for row in result}) == len(photos)
        for subset, size in [('labeled', 40), ('unlabeled', 400)]:
            ids = {row['photo_id'] for row in photos if row['subset'] == subset}
            records = [{key: row[key] for key in ['photo_id', 'boxes', 'error']}
                       for row in result if row['photo_id'] in ids]
            assert len(records) == size
            path = OUT / f'{name}_predictions_{size}.jsonl'
            path.write_text(''.join(json.dumps(row, ensure_ascii=False, allow_nan=False) + '\n' for row in records))
        assert all(row['error'] is None for row in result), f'{name}: исправьте ошибки в предсказаниях в {OUT} и повторите запуск.'
        return result
    finally:
        del models
        gc.collect()
        for gpu in range(2):
            with torch.cuda.device(gpu):
                torch.cuda.empty_cache()




In [11]:
# Метрики
def xyxy(box):
    return [box['left'], box['top'], box['left'] + box['width'], box['top'] + box['height']]


def count_matches(overlap, threshold):
    edges = overlap >= threshold
    owner = {}
    def assign(i, seen):
        for j in np.flatnonzero(edges[i]):
            j = int(j)
            if j in seen:
                continue
            seen.add(j)
            if j not in owner or assign(owner[j], seen):
                owner[j] = i
                return True
        return False
    return sum(assign(i, set()) for i in range(len(edges)))


def metrics(golden, predictions):
    assert len(predictions) == 40 and all(row['error'] is None for row in predictions)
    gold = {row['photo_id']: row['boxes'] for row in golden}
    assert set(gold) == {row['photo_id'] for row in predictions}
    matches = {0.5: 0, 0.75: 0}
    predicted = sum(len(row['boxes']) for row in predictions)
    expected = sum(len(boxes) for boxes in gold.values())
    for row in predictions:
        a = np.array([xyxy(b) for b in gold[row['photo_id']]], dtype=float).reshape(-1, 4)
        b = np.array([xyxy(b) for b in row['boxes']], dtype=float).reshape(-1, 4)
        overlap = pairwise_iou(a, b)
        for threshold in matches:
            matches[threshold] += count_matches(overlap, threshold)
    tp = matches[0.5]
    return {'F1@0.5': 2 * tp / (expected + predicted), 'precision': tp / predicted if predicted else 0,
            'recall': tp / expected, 'FP+FN': predicted + expected - 2 * tp,
            'time, s': sum(row['seconds'] for row in predictions),
            'F1@0.75': 2 * matches[0.75] / (expected + predicted),
            'FP on negatives': sum(len(row['boxes']) for row in predictions if not gold[row['photo_id']])}


In [15]:
# Размер батча на одну GPU.
BATCH_SIZE = 1

# Предсказания
assert type(BATCH_SIZE) is int and BATCH_SIZE > 0, 'BATCH_SIZE должен быть целым положительным числом.'
assert torch.cuda.device_count() == 2, 'Включите T4 ×2.'
assert all('T4' in torch.cuda.get_device_name(i) for i in range(2))
photos, data_dir = load_data()
owl_path, yolo_path, weights = load_weights(data_dir)

configs = {'owl': OWL, 'yolo': YOLO_SETTINGS}
assert all(c['confidence'] >= MIN_CONFIDENCE for c in configs.values()), 'Confidence должен быть не ниже 0,25.'
save(OUT / 'settings.json', {'models': weights, 'configs': configs, 'batch_size_per_gpu': BATCH_SIZE})
(OUT.parent / 'butterfly_results.zip').unlink(missing_ok=True)
prediction_files = [f'{model}_predictions_{size}.jsonl' for model in configs for size in [40, 400]]
for name in ['metrics.csv', 'best_model.json'] + prediction_files:
    (OUT / name).unlink(missing_ok=True)
predictions = {}
for name, config in configs.items():
    predictions[name] = run(name, config, photos, owl_path, yolo_path, BATCH_SIZE)


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

owl:   0%|          | 0/440 [00:00<?, ?фото/s]

yolo:   0%|          | 0/440 [00:00<?, ?фото/s]

In [16]:
# Оценка: эталон читаем после всех предсказаний.
golden = [json.loads(line) for line in (data_dir / 'golden.jsonl').read_text().splitlines()]
assert len(golden) == len({row['photo_id'] for row in golden}) == 40
labeled_ids = {row['photo_id'] for row in photos if row['subset'] == 'labeled'}
assert labeled_ids == {row['photo_id'] for row in golden}
table = pd.DataFrame({name: metrics(golden, [row for row in rows if row['photo_id'] in labeled_ids])
                      for name, rows in predictions.items()}).T
table = table.sort_values(['F1@0.5', 'time, s'], ascending=[False, True])
table.to_csv(OUT / 'metrics.csv', index_label='model')
display(table.round(3))
winner = table.index[0]
save(OUT / 'best_model.json', {'model': winner, 'config': configs[winner]})
print('Лучшая модель по F1@0.5:', winner)


,F1@0.5,precision,recall,FP+FN,"time, s",F1@0.75,FP on negatives
yolo,0.839,0.897,0.788,10.0,25.183,0.645,3.0
owl,0.831,0.844,0.818,11.0,78.273,0.431,1.0


Лучшая модель по F1@0.5: yolo


In [17]:
# Сохранение результатов
archive_path = OUT.parent / 'butterfly_results.zip'
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    files = [OUT / name for name in prediction_files]
    files += [OUT / name for name in ['settings.json', 'metrics.csv', 'best_model.json']]
    for path in files:
        archive.write(path, path.relative_to(OUT))
display(FileLink(str(archive_path)))


/kaggle/working/butterfly_results.zip